<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/UDP_text_sender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 【送信側】文字を UDP で送る 📤

このノートブックは **送る人** が、**受け取る人とは別の Colab** で動かします。
受け取る人が作った **公開アドレス（`udp://……`）** に向けて、文字を送るだけです。

> UDP は「送りっぱなし」の通信です。
> **送った文字が相手に届いたかどうかは、こちら側では分かりません。**
> 届いたかどうかは、受け取る人の画面で確認してもらいます。

実行する順番：**① 送り先を設定 → ② 文字を送る**

## ① 送り先を設定

受け取る人から教わった公開アドレスを、下の `ADDRESS` に貼り付けて実行してください。
（受信側ノートの②で表示された `udp://……` をそのままコピーします）

In [1]:
import re

# ↓ 受け取る人から教わった公開アドレスを貼り付ける
ADDRESS = "udp://bwhxu-136-119-70-15.run.pinggy-free.link:29570" # "udp://ここに貼り付ける.run.pinggy-free.link:00000"

m = re.search(r"([\w.\-]+):(\d+)", ADDRESS.replace("udp://", ""))
if m:
    HOST, PORT = m.group(1), int(m.group(2))
    print(f"送り先：{HOST} : {PORT}")
else:
    HOST = PORT = None
    print("アドレスの形が正しくありません。udp://〜:番号 の形で貼り付けてください。")


送り先：bwhxu-136-119-70-15.run.pinggy-free.link : 29570


## ② 文字を送る

`message` を好きな文字に書き換えて実行してください。
何度でも送れます。

何度か送ると、受け取る人の画面で「届いたり・届かなかったり」する様子が見えるはずです。

In [3]:
# ↓ ここを書き換えて送る
message = "こんにちは UDP"

In [4]:
#実際の送信処理
import socket

tx = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

def send_text(msg):
    if not HOST:
        print("先に①で送り先アドレスを設定してください。")
        return
    data = msg.encode("utf-8")
    tx.sendto(data, (HOST, PORT))      # 送る（送りっぱなし。返事は待たない）
    print(f"送信しました: 「{msg}」 {len(data)} バイト → {HOST}:{PORT}")



send_text(message)


送信しました: 「こんにちは UDP」 19 バイト → bwhxu-136-119-70-15.run.pinggy-free.link:29570


## 連続して送る（到達率の実験）

同じ文字を10回送ります。
受け取る人の画面で「10回のうち何回届いたか」を数えてもらいましょう。
**10回すべては届かない**ことが多いはずです。これが UDP の「届くとは限らない」性質です。

In [5]:
import time

def send_many(n=10):
    if not HOST:
        print("先に①で送り先アドレスを設定してください。")
        return
    for i in range(n):
        tx.sendto(f"test-{i}".encode("utf-8"), (HOST, PORT))
        print(f"  送信 test-{i}")
        time.sleep(0.3)
    print(f"{n} 回送りました。何回届いたかは受け取る人に数えてもらってください。")

send_many(10)


  送信 test-0
  送信 test-1
  送信 test-2
  送信 test-3
  送信 test-4
  送信 test-5
  送信 test-6
  送信 test-7
  送信 test-8
  送信 test-9
10 回送りました。何回届いたかは受け取る人に数えてもらってください。


### まとめ（送信側）

- 受け取る人の公開アドレスへ、文字を送るだけ。
- 送れても、届いたかどうかはこちらでは分からない。
- 確実に届けたいなら TCP を使うか、「届いたか確認して送り直す」仕組みが必要。これが UDP と TCP の違い。